In [2]:
import pandas as pd

# 1️⃣ 데이터 로드
df = pd.read_csv("data/1_2_(로우데이터_합본.csv)목적별_국적별_입국(05년1월~25년5월).csv", encoding='cp949')

# 2️⃣ 불필요 row 제거
df = df[
    (~df['국적'].str.endswith('주', na=False)) &
    (~df['국적'].str.endswith('기타', na=False)) &
    (~df['국적'].isin(['소 계', '전 체', '미상', '기 타', '교포', '중 동'])) &
    (~df['목적'].isin(['소 계', '전 체']))
]
df = df[df['국적'].notna()]

# 3️⃣ 날짜(연월) 컬럼만 추출, 2025-05까지
fixed_cols = ['국적', '목적']
date_cols = [col for col in df.columns if ('년' in col and '월' in col)]
date_cols_final = []
for c in date_cols:
    try:
        year = int(c[:4])
        month = int(c[5:7])
        if (year < 2025) or (year == 2025 and month <= 5):
            date_cols_final.append(c)
    except:
        continue

cols_to_use = fixed_cols + date_cols_final
df = df[cols_to_use]

# 4️⃣ 숫자 변환
for col in date_cols_final:
    df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# 5️⃣ long-form 변환
long_df = df.melt(id_vars=fixed_cols, value_vars=date_cols_final,
                  var_name='ym', value_name='입국자수')

# 6️⃣ '국적'/'목적' 모든 공백 제거
long_df['국적'] = long_df['국적'].str.replace(' ', '', regex=False)
long_df['목적'] = long_df['목적'].str.replace(' ', '', regex=False)

# 7️⃣ 'ym'에서 년, 월 추출
long_df['년'] = long_df['ym'].str.extract(r'(\d{4})').astype(int)
long_df['월'] = long_df['ym'].str.extract(r'(\d{1,2})월').astype(int)

# 8️⃣ 정렬 및 결측/음수 제거
long_df = long_df[['국적', '목적', '년', '월', '입국자수']]
long_df = long_df[long_df['입국자수'].notnull() & (long_df['입국자수'] >= 0)]

# 9️⃣ 저장
long_df.to_csv("data/목적별국적별입국_정제_2025년5월까지_공백제거.csv", index=False, encoding='cp949')
print("✔️ 띄어쓰기 없는 국적/목적으로 2025년5월까지 저장 완료!")


✔️ 띄어쓰기 없는 국적/목적으로 2025년5월까지 저장 완료!
